In [1]:
import osmnx as ox
from ipyleaflet import Map, Marker, CircleMarker
from ipywidgets import Button, VBox
from IPython.display import display


def astana_parking_system(radius_m=1000):

    center = (51.1282, 71.4304)
    m = Map(center=center, zoom=13)

    selected_points = []
    parking_list = []

    print("Click on the map:")
    print("1st click = A")
    print("2nd click = B")
    print("3rd click = C (optional, otherwise C = A)")

    # ---- Click handler ----
    def handle_click(**kwargs):
        if kwargs.get("type") == "click":
            latlng = kwargs.get("coordinates")
            selected_points.append(latlng)

            marker = Marker(location=latlng)
            m.add_layer(marker)

            print(f"Point {len(selected_points)}:", latlng)

    m.on_interaction(handle_click)

    # ---- Finish button ----
    finish_button = Button(description="Finish Selection")

    def process(b):

        if len(selected_points) < 2:
            print("You must click at least 2 points (A and B)")
            return

        parking_list.clear()

        A = selected_points[0]
        B = selected_points[1]
        C = selected_points[2] if len(selected_points) >= 3 else A

        print("\nSaved Points:")
        print("A:", A)
        print("B:", B)
        print("C:", C)

        print("\nSearching for ALL parking near B...")

        tags = {"amenity": "parking"}

        parking_gdf = ox.features_from_point(
            B,
            tags=tags,
            dist=radius_m
        )

        print("Geometry types found:")
        print(parking_gdf.geometry.geom_type.value_counts())

        # ---- Convert everything to coordinates ----
        for _, row in parking_gdf.iterrows():

            geom = row.geometry

            if geom.geom_type == "Point":
                lat = geom.y
                lon = geom.x

            elif geom.geom_type in ["Polygon", "MultiPolygon"]:
                centroid = geom.centroid
                lat = centroid.y
                lon = centroid.x

            else:
                continue

            parking_list.append((lat, lon))

            circle = CircleMarker(
                location=(lat, lon),
                radius=2,
                color="blue",
                fill=True
            )
            m.add_layer(circle)

        print(f"\nTotal parking objects found: {len(parking_list)}")
        print("First 10 parking coordinates:")
        print(parking_list[:10])

        astana_parking_system.data = {
            "A": A,
            "B": B,
            "C": C,
            "parking_list": parking_list
        }

    finish_button.on_click(process)

    display(VBox([m, finish_button]))


In [2]:
astana_parking_system(1000)

Click on the map:
1st click = A
2nd click = B
3rd click = C (optional, otherwise C = A)


In [3]:
G = ox.graph_from_place("Astana, Kazakhstan", network_type='drive')
G

In [4]:
len(G.nodes)

8494

In [5]:
len(G.edges)

21062

In [6]:
list(G.edges(data=True))[1]

(250783684,
 2439196353,
 {'osmid': [1459982272, 1459982256, 1459982257, 1459982258, 1459982265],
  'highway': 'residential',
  'lanes': '4',
  'name': 'Ғабдолла Тоқай көшесі',
  'oneway': False,
  'reversed': True,
  'length': np.float64(240.52039037290308),
  'geometry': <LINESTRING (71.412 51.12, 71.412 51.12, 71.411 51.12, 71.41 51.12, 71.41 51...>})

In [7]:
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)


## ROUTING

### Initialization of graph

In [1]:
import networkx as nx # Graph logic: routing, shortest paths on road networks
import osmnx as ox # Download real road networks from OpenStreetMap
import os # File and path handling (check/save/load files)
import pickle # Save/load Python objects (cache graphs, data)
import math # Geometry and numeric calculations (angles, distances)
from functools import lru_cache # Cache function results to avoid recomputation
from shapely.geometry import Point, LineString # Represent locations as map points
from shapely.ops import nearest_points # Find closest geometry (e.g., spot → road)

# ==========================================
# CONFIGURATION
# ==========================================
PLACE_NAME = "Astana, Kazakhstan"
GRAPH_FILENAME = "astana_drive.graphml"

# PENALTIES (Seconds)
# COST_U_TURN = 300.0   
# COST_LEFT = 45.0      
# COST_RIGHT = 10.0     
# COST_STRAIGHT = 0.0
COST_U_TURN = 0.0   
COST_LEFT = 0.0      
COST_RIGHT = 0.0     
COST_STRAIGHT = 0.0

G = None      
G_TURN = None 

def initialize_graph(): #запуск карты
    global G, G_TURN 
    if G is not None: return #Если карта уже есть -> ничего не делаем.
    if os.path.exists(GRAPH_FILENAME):
        G = ox.load_graphml(GRAPH_FILENAME)
    else:
        G = ox.graph_from_place(PLACE_NAME, network_type='drive')
        ox.save_graphml(G, GRAPH_FILENAME)
    
    # Add travel times if missing
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)
    G = ox.add_edge_bearings(G)

In [2]:
initialize_graph()

In [3]:
def get_unique_edge_names(G):
    names = set()

    for _, _, _, data in G.edges(keys=True, data=True):
        name = data.get("name")
        if not name:
            continue

        if isinstance(name, list):
            for n in name:
                names.add(n.strip())
        else:
            names.add(name.strip())

    return sorted(names)

unique_names = get_unique_edge_names(G)
print(len(unique_names))
print(unique_names[:50])

1377
['1-й Алматинский переулок', '1-й Разина пер.', '1-я Алматинская улица', '101-я улица', '126 улица', '135-я улица', '150 лет Абая', '163-я улица', '172', '2-й Алматинский пер.', '2-й Разина пер.', '2-я Нагорная', '22 улица', '23-ші өткел', '25-ші көше', '27 улица', '28 улица', '28/5 көше', '29/2', '30 улица', '39-шы көше', '41 улица', '44-я', '49 улица', '69 улица', '70 лет Октября көшесі', '78 улица', '85', '85-я улица', '92-я', 'E 12', 'E 13', 'E 15', 'E 26', 'E 27', 'E 28', 'E 495', 'E-900', 'Ілияс Жансүгіров көшесі', 'Інжір көшесі', 'А 102', 'А 105', 'А 108', 'А 184', 'А 194', 'А 207', 'А-52 көшесі', 'А32', 'А355', 'А356']


In [4]:
def inspect_name_types(G):
    types = set()
    examples = {}

    for _, _, _, data in G.edges(keys=True, data=True):
        name = data.get("name")

        if name is None:
            types.add("None")
        elif isinstance(name, list):
            types.add("list")
            examples.setdefault("list", name)
        elif isinstance(name, str):
            types.add("str")
            examples.setdefault("str", name)
        else:
            types.add(type(name).__name__)
            examples.setdefault(type(name).__name__, name)

    return types, examples


types, examples = inspect_name_types(G)
print("Types found:", types)
print("Examples:", examples)

Types found: {'list', 'str', 'None'}
Examples: {'str': 'Қабанбай Батыр даңғылы', 'list': ['улица Гете', 'проспект Республики']}


In [12]:
def print_traffic_signal_nodes():
    signal_nodes = []

    for node, data in G.nodes(data=True):
        if data.get("highway") == "traffic_signals":
            signal_nodes.append((node, data))

    print(f"Total traffic signals found: {len(signal_nodes)}\n")

    for node_id, attrs in signal_nodes:
        print(f"Node ID: {node_id}")
        print(f"  Coordinates: ({attrs.get('y')}, {attrs.get('x')})")
        print(f"  Full data: {attrs}")
        print("-" * 40)

In [13]:
print_traffic_signal_nodes()

Total traffic signals found: 457

Node ID: 259015127
  Coordinates: (51.1815454, 71.3741971)
  Full data: {'y': 51.1815454, 'x': 71.3741971, 'highway': 'traffic_signals', 'street_count': 4}
----------------------------------------
Node ID: 259987030
  Coordinates: (51.0419123, 71.4419443)
  Full data: {'y': 51.0419123, 'x': 71.4419443, 'highway': 'traffic_signals', 'street_count': 4}
----------------------------------------
Node ID: 259993125
  Coordinates: (51.2045533, 71.4901081)
  Full data: {'y': 51.2045533, 'x': 71.4901081, 'highway': 'traffic_signals', 'street_count': 3}
----------------------------------------
Node ID: 259996504
  Coordinates: (51.1558628, 71.4732684)
  Full data: {'y': 51.1558628, 'x': 71.4732684, 'highway': 'traffic_signals', 'street_count': 4}
----------------------------------------
Node ID: 259996554
  Coordinates: (51.1633372, 71.471845)
  Full data: {'y': 51.1633372, 'x': 71.471845, 'highway': 'traffic_signals', 'street_count': 3}
------------------------

In [14]:
tags = {"highway": "traffic_signals"}
signals = ox.features_from_place(PLACE_NAME, tags)
print(signals.head())
print("Total signals:", len(signals))

                                    geometry          highway traffic_signals  \
element id                                                                      
node    259015127   POINT (71.3742 51.18155)  traffic_signals             NaN   
        259987030  POINT (71.44194 51.04191)  traffic_signals             NaN   
        259993125  POINT (71.49011 51.20455)  traffic_signals             NaN   
        259996504  POINT (71.47327 51.15586)  traffic_signals             NaN   
        259996554  POINT (71.47184 51.16334)  traffic_signals             NaN   

                  crossing button_operated traffic_signals:sound  \
element id                                                         
node    259015127      NaN             NaN                   NaN   
        259987030      NaN             NaN                   NaN   
        259993125      NaN             NaN                   NaN   
        259996504      NaN             NaN                   NaN   
        259996554      N

In [15]:
missing = []

for u, v, k, data in G.edges(keys=True, data=True):
    if "name" not in data or not data["name"]:
        missing.append((u, v, k))

print("Edges without street name:", len(missing))

Edges without street name: 6914


In [22]:
keywords = [
    "Тұран", "Қорғалжын", "Достық", "Ақмешіт", "Сарайшық",
    "Түркістан", "Қабанбай", "Қонаев", "Сауран",
    "Тәуелсіздік", "Момышұлы", "Кенесары",
    "Жеңіс", "Республика", "Сейфуллин", "Уәлиханов", "Коргалжын", "Бауыржан", "Момышулы","Кеңесары", "Женис","Республик", "Уалихан", "Шокан", 
    "Мангилик", "Мәңгілік", "Мухамедхан", "Мұхамедхан", "Сыганак", "Сығанақ", "Арай", "Құдайберді", "Кудайберд"
]

matched_names = set()

for _, _, _, data in G.edges(keys=True, data=True):
    name = data.get("name")
    if not name:
        continue

    names = name if isinstance(name, list) else [name]

    for n in names:
        for kw in keywords:
            if kw.lower() in n.lower():
                matched_names.add(n)

print("Matched street names:")
for n in sorted(matched_names):
    print(n)

Matched street names:
Арай көшесі
Ақмешіт көшесі
Бауыржан Момышұлы даңғылы
Достық көшесі
Коргалжын шоссе
Мухамедханова
Мәңгілік Ел даңғылы
Сарайшық көшесі
Сауран көшесі
Сығанақ көшесі
Түркістан көшесі
Тұран даңғылы
Тәуелсіздік даңғылы
Шәкәрім Құдайбердіұлы даңғылы
проспект Бауыржана Момышулы
проспект Женис
проспект Республики
улица  Сейфуллина
улица Кенесары
улица Сакена Сейфуллина
улица Сарайшык
улица Сейфуллина
улица Шокана Валиханова
Қабанбай Батыр даңғылы
Қайым Мұхамедханов көшесі
Қонаев көшесі


In [23]:
highway_values = set()

for _, data in G.nodes(data=True):
    if "highway" in data:
        highway_values.add(data["highway"])

print(highway_values)

{'traffic_signals;crossing', 'traffic_signals', 'milestone', 'motorway_junction', 'crossing'}


In [24]:
from collections import Counter

counter = Counter()

for _, data in G.nodes(data=True):
    if "highway" in data:
        counter[data["highway"]] += 1

print(counter)

Counter({'traffic_signals': 457, 'motorway_junction': 17, 'crossing': 10, 'milestone': 1, 'traffic_signals;crossing': 1})


In [3]:
type(G)

networkx.classes.multidigraph.MultiDiGraph

In [4]:
len(G.nodes), len(G.edges)

(8469, 21030)

In [5]:
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 8469
Edges: 21030


In [6]:
type(G.nodes), type(G.edges), type(G.adj), type(G.graph)

(networkx.classes.reportviews.NodeView,
 networkx.classes.reportviews.OutMultiEdgeView,
 networkx.classes.coreviews.MultiAdjacencyView,
 dict)

In [7]:
G.graph

{'created_date': '2026-01-16 16:46:09',
 'created_with': 'OSMnx 2.0.7',
 'crs': 'epsg:4326',
 'simplified': True}

In [15]:
dir(G)

['__class__',
 '__contains__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__networkx_backend__',
 '__networkx_cache__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_adj',
 '_node',
 '_pred',
 '_succ',
 'add_edge',
 'add_edges_from',
 'add_node',
 'add_nodes_from',
 'add_weighted_edges_from',
 'adj',
 'adjacency',
 'adjlist_inner_dict_factory',
 'adjlist_outer_dict_factory',
 'clear',
 'clear_edges',
 'copy',
 'degree',
 'edge_attr_dict_factory',
 'edge_key_dict_factory',
 'edge_subgraph',
 'edges',
 'get_edge_data',
 'graph',
 'graph_attr_dict_factory',
 'has_edge',
 'has_node',
 'has_predecessor',
 'has_successor',

In [8]:
list(G.nodes)[:5]

[250783684, 250783685, 259015127, 259015326, 259016376]

In [9]:
list(G.nodes(data=True))[:5]

[(250783684, {'y': 51.1200654, 'x': 71.4120309, 'street_count': 3}),
 (250783685, {'y': 51.1364467, 'x': 71.4157077, 'street_count': 4}),
 (259015127,
  {'y': 51.1815454,
   'x': 71.3741971,
   'highway': 'traffic_signals',
   'street_count': 4}),
 (259015326, {'y': 51.2043639, 'x': 71.3104988, 'street_count': 3}),
 (259016376, {'y': 51.1213959, 'x': 71.6480844, 'street_count': 4})]

In [18]:
list(G.edges(data=True))[1]

(250783684,
 2439196353,
 {'osmid': [1459982272, 1459982256, 1459982257, 1459982258, 1459982265],
  'highway': 'residential',
  'lanes': '4',
  'name': 'Ғабдолла Тоқай көшесі',
  'oneway': False,
  'reversed': True,
  'length': 240.52039037290308,
  'geometry': <LINESTRING (71.412 51.12, 71.412 51.12, 71.411 51.12, 71.41 51.12, 71.41 51...>,
  'speed_kph': 55.493482309124765,
  'travel_time': 15.603154988889136,
  'bearing': 284.1684277952138})

In [36]:
import folium

node_id = 250783684
lat = G.nodes[node_id]['y']
lon = G.nodes[node_id]['x']

m = folium.Map(location=[lat, lon], zoom_start=16)
folium.Marker([lat, lon]).add_to(m)
m


### Augmentation of Graph with Spots

Each spot must be a dictionary like this:


{
    "id": "manual_12345",          # unique string
    "coords": (lat, lon),          # tuple of floats
    "phi_exit_seconds": 60,        # search penalty (optional but used later)
    "p_i": 0.8                     # success probability (optional but used later)
}

In [17]:
def augment_graph_with_spots(spots):
    """
    Realizes the 'Split-Edge' theory.
    Transforms the graph by injecting spot-nodes into existing edges.
    """
    global G
    initialize_graph()  # убеждаемся, что карта дорог загружена

    # Work on a copy to keep the base graph clean
    G_aug = G.copy()

    # 1. Find nearest edges for all spots
    lats = [s["coords"][0] for s in spots]
    lngs = [s["coords"][1] for s in spots]
    nearest_edges = ox.nearest_edges(
        G_aug, lngs, lats
    )  # На какой именно улице она лежит ближе всего?

    # 2. Group spots by the edge they belong to
    edge_to_spots = {}
    for i, edge in enumerate(nearest_edges):
        if edge not in edge_to_spots:
            edge_to_spots[edge] = []
        edge_to_spots[edge].append(spots[i])

    for (u, v, k), spot_list in edge_to_spots.items():
        edge_data = G_aug.get_edge_data(u, v, k)
        if not edge_data:
            continue

        edge_geom = edge_data.get("geometry", None)  ### we have None values too
        total_len = edge_data["length"]
        total_time = edge_data["travel_time"]

        # Calculate distance along edge for each spot
        spot_offsets = []
        for s in spot_list:
            p = Point(s["coords"][1], s["coords"][0])
            # if edge_geom:
            #     dist_along = edge_geom.project(p)
            # else:
            #     # Fallback to simple linear interp if no geometry
            #     dist_along = 0  # Simplified
            ##### CHANGE ###############
            if edge_geom is None:
                # build straight-line geometry from u to v
                u_x, u_y = G_aug.nodes[u]["x"], G_aug.nodes[u]["y"]
                v_x, v_y = G_aug.nodes[v]["x"], G_aug.nodes[v]["y"]
                edge_geom = LineString([(u_x, u_y), (v_x, v_y)])
            dist_along = edge_geom.project(p)
            ##### CHANGE ###############
            spot_offsets.append((s["id"], dist_along, s["coords"]))

        # Sort spots by distance along the edge
        spot_offsets.sort(key=lambda x: x[1])

        # 3. Split the edge
        curr_u = u
        last_offset = 0

        for spot_id, offset, coords in spot_offsets:
            new_node = f"spot_{spot_id}"
            ###### CHANGE 1 ##########
            proj_point = edge_geom.interpolate(offset)

            G_aug.add_node(
                new_node,
                x=proj_point.x,
                y=proj_point.y,
            )
            # G_aug.add_node(new_node, x=coords[1], y=coords[0])

            ###### CHANGE 1 ##########

            # Calculate segment metrics
            seg_len = offset - last_offset
            ratio = seg_len / total_len if total_len > 0 else 0
            seg_time = total_time * ratio

            # Add the segment from current position to this spot
            G_aug.add_edge(
                curr_u,
                new_node,
                length=seg_len,
                travel_time=seg_time,
                highway=edge_data.get("highway"),
            )

            curr_u = new_node
            last_offset = offset

        # Final segment to the original end intersection
        final_len = total_len - last_offset
        ratio = final_len / total_len if total_len > 0 else 0
        G_aug.add_edge(
            curr_u,
            v,
            length=final_len,
            travel_time=total_time * ratio,
            highway=edge_data.get("highway"),
        )

        # Remove the original long edge
        G_aug.remove_edge(u, v, k)

    return G_aug


edge_geom is a LINESTRING representing the road curve.

project(p) computes:

distance from the start of the LINESTRING to the perpendicular projection of point 
𝑝
distance from the start of the LINESTRING to the perpendicular projection of point p

In [18]:
G.get_edge_data(250783684,
 2439196317,0)

{'osmid': [535921238, 1456296679],
 'highway': 'secondary',
 'lanes': '2',
 'maxspeed': '60',
 'name': 'Қабанбай Батыр даңғылы',
 'oneway': True,
 'reversed': False,
 'length': 192.23034378086237,
 'geometry': <LINESTRING (71.412 51.12, 71.412 51.12, 71.412 51.119, 71.412 51.119, 71.41...>,
 'speed_kph': 60.0,
 'travel_time': 11.533820626851742,
 'bearing': 186.1094103623889}

### Build full Parking Dataset for Astana

In [19]:
import osmnx as ox
import json
from shapely.geometry import Point

def build_astana_spots(
    output_file="astana_spots.json",
    deduplicate=True,
    min_distance_m=5
):
    """
    Downloads all parking objects from OSM for Astana,
    converts them into internal spot format,
    optionally removes near-duplicate coordinates,
    and saves to JSON.
    """

    print("Downloading parking data from OSM...")

    tags = {
        "amenity": ["parking", "parking_space"],
        "parking": True
    }

    gdf = ox.features_from_place(
        "Astana, Kazakhstan",
        tags=tags
    )

    print("\nGeometry types found:")
    print(gdf.geometry.geom_type.value_counts())

    # Keep only geometries we can convert
    gdf = gdf[gdf.geometry.notnull()]

    spots = []
    coords_seen = []

    counter = 0

    for _, row in gdf.iterrows():
        geom = row.geometry

        # Convert to coordinate
        if geom.geom_type == "Point":
            lat = geom.y
            lon = geom.x

        elif geom.geom_type in ["Polygon", "MultiPolygon"]:
            centroid = geom.centroid
            lat = centroid.y
            lon = centroid.x

        else:
            continue

        # Optional deduplication
        if deduplicate:
            point = Point(lon, lat)
            duplicate = False
            for existing in coords_seen:
                if point.distance(existing) < min_distance_m / 111000:  # rough meter->degree
                    duplicate = True
                    break
            if duplicate:
                continue
            coords_seen.append(point)

        spot = {
            "id": f"osm_{counter}",
            "coords": [float(lat), float(lon)],  # JSON-safe
            "phi_exit_seconds": 0,
            "p_i": 0.5
        }

        spots.append(spot)
        counter += 1

    print(f"\nTotal parking spots created: {len(spots)}")

    with open(output_file, "w") as f:
        json.dump(spots, f, indent=2)

    print(f"Saved to {output_file}")

    return spots


In [20]:
spots = build_astana_spots()
spots


Geometry types found:
Polygon         992
Point           207
MultiPolygon      2
Name: count, dtype: int64

Total parking spots created: 1199
Saved to astana_spots.json


[{'id': 'osm_0',
  'coords': [51.159674, 71.4808744],
  'phi_exit_seconds': 0,
  'p_i': 0.5},
 {'id': 'osm_1',
  'coords': [51.1566613, 71.56731],
  'phi_exit_seconds': 0,
  'p_i': 0.5},
 {'id': 'osm_2',
  'coords': [51.1548371, 71.5720441],
  'phi_exit_seconds': 0,
  'p_i': 0.5},
 {'id': 'osm_3',
  'coords': [51.2007292, 71.3517671],
  'phi_exit_seconds': 0,
  'p_i': 0.5},
 {'id': 'osm_4',
  'coords': [51.2006116, 71.3502328],
  'phi_exit_seconds': 0,
  'p_i': 0.5},
 {'id': 'osm_5',
  'coords': [51.1999949, 71.4331035],
  'phi_exit_seconds': 0,
  'p_i': 0.5},
 {'id': 'osm_6',
  'coords': [51.1270569, 71.4255625],
  'phi_exit_seconds': 0,
  'p_i': 0.5},
 {'id': 'osm_7',
  'coords': [51.1289902, 71.4208268],
  'phi_exit_seconds': 0,
  'p_i': 0.5},
 {'id': 'osm_8',
  'coords': [51.1302576, 71.4163342],
  'phi_exit_seconds': 0,
  'p_i': 0.5},
 {'id': 'osm_9',
  'coords': [51.1307707, 71.4165216],
  'phi_exit_seconds': 0,
  'p_i': 0.5},
 {'id': 'osm_10',
  'coords': [51.1333514, 71.4215547

In [21]:
G_aug = augment_graph_with_spots(spots)

spot_nodes = [n for n in G_aug.nodes if str(n).startswith("spot_")]

print("Total spot nodes in graph:", len(spot_nodes))
print("Example spot nodes:", spot_nodes[:5])


Total spot nodes in graph: 1199
Example spot nodes: ['spot_osm_563', 'spot_osm_422', 'spot_osm_0', 'spot_osm_1', 'spot_osm_2']


In [42]:
#!pip install matplotlib

In [47]:
import osmnx as ox
import folium

# Build augmented graph
G_aug = augment_graph_with_spots(spots)

# Convert edges to GeoDataFrame
edges = ox.graph_to_gdfs(G_aug, nodes=False)


In [48]:
# Center map roughly on Astana
center_lat = sum(s["coords"][0] for s in spots) / len(spots)
center_lon = sum(s["coords"][1] for s in spots) / len(spots)

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles="cartodbpositron"
)


In [49]:
# Center map roughly on Astana
center_lat = sum(s["coords"][0] for s in spots) / len(spots)
center_lon = sum(s["coords"][1] for s in spots) / len(spots)

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles="cartodbpositron"
)


In [50]:
for s in spots:
    lat, lon = s["coords"]
    folium.CircleMarker(
        location=[lat, lon],
        radius=3,                 # small point
        color="#6a0dad",           # purple
        fill=True,
        fill_color="#6a0dad",
        fill_opacity=0.9,
        opacity=0.9,
        popup=f"Spot {s['id']}"
    ).add_to(m)


In [51]:
for s in spots:
    lat, lon = s["coords"]
    folium.CircleMarker(
        location=[lat, lon],
        radius=3,                 # small point
        color="#6a0dad",           # purple
        fill=True,
        fill_color="#6a0dad",
        fill_opacity=0.9,
        opacity=0.9,
        popup=f"Spot {s['id']}"
    ).add_to(m)


In [52]:
m.save("augmented_astana_map.html")

## check how G_aug looks like

In [23]:
G_aug = augment_graph_with_spots(spots)


In [22]:
spot_nodes = []
road_nodes = []

for n, data in G_aug.nodes(data=True):
    if str(n).startswith("spot_"):
        spot_nodes.append((n, data))
    else:
        road_nodes.append((n, data))


In [27]:
import osmnx as ox

edges = ox.graph_to_gdfs(G_aug, nodes=False)


In [28]:
import folium

# Center map on mean of spot coords
lat = sum(s["coords"][0] for s in spots) / len(spots)
lon = sum(s["coords"][1] for s in spots) / len(spots)

m = folium.Map(
    location=[lat, lon],
    zoom_start=14,
    tiles="cartodbpositron"
)


In [29]:
for node_id, data in spot_nodes:
    folium.CircleMarker(
        location=[data["y"], data["x"]],
        radius=3,
        color="purple",
        fill=True,
        fill_opacity=1.0,
        popup=str(node_id),
    ).add_to(m)


In [58]:
for node_id, data in road_nodes[:2000]:  # limit!
    if "x" in data and "y" in data:
        folium.CircleMarker(
            location=[data["y"], data["x"]],
            radius=1,
            color="blue",
            opacity=0.3,
        ).add_to(m)


In [59]:
m.save("G_aug_structure.html")

In [30]:
import json
from ipyleaflet import Map, CircleMarker
from IPython.display import display

# --- Load spots from JSON ---
with open("astana_spots.json", "r") as f:
    spots = json.load(f)

print(f"Loaded {len(spots)} parking spots")

# --- Create map centered on Astana ---
center = (51.1282, 71.4304)
m = Map(center=center, zoom=12)

# --- Add spots to map ---
for s in spots:
    lat, lon = s["coords"]

    marker = CircleMarker(
        location=(lat, lon),
        radius=2,
        color="blue",
        fill=True
    )

    m.add_layer(marker)

display(m)


Loaded 1199 parking spots


Map(center=[51.1282, 71.4304], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…

In [24]:
test_spots = spots[:200]   # start small

G_aug = augment_graph_with_spots(test_spots)

print("\nAugmented graph:")
print("Nodes:", len(G_aug.nodes))
print("Edges:", len(G_aug.edges))


Augmented graph:
Nodes: 8669
Edges: 21230


In [25]:
print("Nodes:", len(G.nodes))
print("Edges:", len(G.edges))

Nodes: 8469
Edges: 21030


In [26]:
spot_nodes = [n for n in G_aug.nodes if str(n).startswith("spot_")]
print("Inserted spot nodes:", len(spot_nodes))


Inserted spot nodes: 200


In [27]:
example_spot_node = spot_nodes[0]
print("Example spot node:", example_spot_node)
print("Attributes:", G_aug.nodes[example_spot_node])


Example spot node: spot_osm_0
Attributes: {'x': 71.48077574215387, 'y': 51.159768030517114}


In [35]:
first_spot = test_spots[0]
print(first_spot)


{'id': 'osm_0', 'coords': [51.159674, 71.4808744], 'phi_exit_seconds': 0, 'p_i': 0.5}


In [36]:
import networkx as nx

start_node = list(G_aug.nodes)[0]
path = nx.shortest_path(G_aug, start_node, example_spot_node, weight="travel_time")
print("Path length:", len(path))


Path length: 46


## GET ROUTE FUNCTION 

In [100]:
def _bearing(lat1, lon1, lat2, lon2):
    dLon = math.radians(lon2 - lon1)
    lat1 = math.radians(lat1)
    lat2 = math.radians(lat2)

    y = math.sin(dLon) * math.cos(lat2)
    x = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(
        dLon
    )

    brng = math.degrees(math.atan2(y, x))
    return (brng + 360) % 360


def get_route(u_coords, v_coords, custom_G=None):
    target_G = custom_G if custom_G else G

    # --- Resolve u ---
    if isinstance(u_coords, str):
        u_node = u_coords
    else:
        u_node = ox.nearest_nodes(target_G, u_coords[1], u_coords[0])

    # --- Resolve v ---
    if isinstance(v_coords, str):
        v_node = v_coords
    else:
        v_node = ox.nearest_nodes(target_G, v_coords[1], v_coords[0])

    if isinstance(u_node, str) and u_node not in target_G.nodes:
        raise KeyError(f"u_node {u_node} not found in graph")

    if isinstance(v_node, str) and v_node not in target_G.nodes:
        raise KeyError(f"v_node {v_node} not found in graph")

    try:
        path = nx.shortest_path(target_G, u_node, v_node, weight="travel_time")

        path_coords = [[target_G.nodes[n]["y"], target_G.nodes[n]["x"]] for n in path]

        time = 0.0
        for i in range(len(path) - 1):
            edge_data = target_G.get_edge_data(path[i], path[i + 1])
            if isinstance(edge_data, dict):
                edge_data = list(edge_data.values())[0]
            time += edge_data.get("travel_time", 1.0)
        # --- Turn penalties ---
        turn_penalty = 0.0

        for i in range(1, len(path) - 1):
            n1, n2, n3 = path[i - 1], path[i], path[i + 1]

            lat1, lon1 = target_G.nodes[n1]["y"], target_G.nodes[n1]["x"]
            lat2, lon2 = target_G.nodes[n2]["y"], target_G.nodes[n2]["x"]
            lat3, lon3 = target_G.nodes[n3]["y"], target_G.nodes[n3]["x"]

            b1 = _bearing(lat1, lon1, lat2, lon2)
            b2 = _bearing(lat2, lon2, lat3, lon3)

            angle = (b2 - b1 + 180) % 360 - 180

            if abs(angle) < 20:
                turn_penalty += COST_STRAIGHT
            elif abs(angle) > 150:
                turn_penalty += COST_U_TURN
            elif angle > 0:
                turn_penalty += COST_RIGHT
            else:
                turn_penalty += COST_LEFT
        return {
            "path": path_coords,
            "travel_time": time + turn_penalty,
            "nodes": path,
        }  ### added nodes key and value
    except Exception:
        return {"path": [], "travel_time": float("inf"), "nodes": []}


In [107]:
def get_route(u_coords, v_coords, custom_G=None):
    target_G = custom_G if custom_G else G

    # --- Resolve u ---
    if isinstance(u_coords, str):
        u_node = u_coords
    else:
        u_node = ox.nearest_nodes(target_G, u_coords[1], u_coords[0])

    # --- Resolve v ---
    if isinstance(v_coords, str):
        v_node = v_coords
    else:
        v_node = ox.nearest_nodes(target_G, v_coords[1], v_coords[0])

    if isinstance(u_node, str) and u_node not in target_G.nodes:
        raise KeyError(f"u_node {u_node} not found in graph")

    if isinstance(v_node, str) and v_node not in target_G.nodes:
        raise KeyError(f"v_node {v_node} not found in graph")

    try:
        path = nx.shortest_path(target_G, u_node, v_node, weight="travel_time")

        path_coords = [[target_G.nodes[n]["y"], target_G.nodes[n]["x"]] for n in path]

        time = 0.0
        for i in range(len(path) - 1):
            edge_data = target_G.get_edge_data(path[i], path[i + 1])
            if isinstance(edge_data, dict):
                edge_data = list(edge_data.values())[0]
            time += edge_data.get("travel_time", 1.0)

        return {
            "path": path_coords,
            "travel_time": time,
            "nodes": path,
        }  ### added nodes key and value

    except Exception:
        return {"path": [], "travel_time": float("inf"), "nodes": []}


In [108]:
start = (51.1282, 71.4304)
end   = (51.1200, 71.4150)

result = get_route(start, end, G_aug)

print("Travel time (sec):", result["travel_time"])
print("Path length:", len(result["path"]))
print("First 5 coords:", result["path"][:5])

Travel time (sec): 127.63619573755646
Path length: 13
First 5 coords: [[51.1280767, 71.4284316], [51.1262959, 71.4277305], [51.1262205, 71.4277009], [51.1236422, 71.4266859], [51.1242032, 71.42303]]


## Transformations 

### StateAdapter

In [109]:
import json  # Read/write structured data (spots, configs, results)
import os
from geopy.distance import geodesic  # Compute real-world distances between coordinates
import math
import routing  # Project routing logic (paths, travel times, graph access)


class StateAdapter:
    def __init__(
        self,
        start,
        dest,
        ref,
        spots,
        traffic_multiplier=1.4,
        distance_cache=None,
        graph=None,
    ):
        self.start, self.dest, self.ref = (
            tuple(start),
            tuple(dest),
            tuple(ref),
        )  # coordinates saved in tuples to avoid changes
        self.spots = spots
        self.traffic_multiplier = traffic_multiplier
        # This cache now stores BASE travel time (without traffic multiplier)
        self.distance_cache = (
            distance_cache if distance_cache is not None else {}
        )  # distance cashe is given as dict
        # Key = pair of coordinates; Value = base travel time
        self.graph = graph if graph is not None else routing.G
        # Astana Longitude Scaling (51.1N)
        self.lng_scale = math.cos(math.radians(self.start[0]))
        self.axis_b = self.dest  # for the direction of movement

        self.spot_topology = []
        self.node_cache = {
            "start": self._map_to_avenue(self.start),  # 0,0
            "ref": self._map_to_avenue(self.ref),  # |AB|, 0
            "dest": self._map_to_avenue(self.dest),
        }

        for s in self.spots:
            progress, side_val = self._map_to_avenue(s["coords"])  # for point P
            s["_cached_topo"] = (
                progress,
                side_val,
            )  # cached topo for spot (creates new key and value)
            self.spot_topology.append(
                {"progress": progress, "side": "LEFT" if side_val > 0 else "RIGHT"}
            )

        self.start_progress = self.node_cache["start"][0]
        print(
            f"[Topology] Mapped {len(self.spots)} spots. Cache loaded with {len(self.distance_cache)} routes."
        )

    def _map_to_avenue(self, point):
        yA, xA = self.start[0], self.start[1] * self.lng_scale  # point of start
        yB, xB = self.axis_b[0], self.axis_b[1] * self.lng_scale  # point of ref
        yP, xP = point[0], point[1] * self.lng_scale  # point of parking

        side_val = (xP - xA) * (yB - yA) - (yP - yA) * (
            xB - xA
        )  # determinant of AP, AB (to the LEFT or RIGHT from the road to AB)
        dx_ab, dy_ab = xB - xA, yB - yA  # A->B
        dx_ap, dy_ap = xP - xA, yP - yA  # A->P
        mag_ab = math.sqrt(dx_ab**2 + dy_ab**2)  # magnitutde of AB |AB|
        if mag_ab == 0:
            return 0, 0
        progress = (dx_ap * dx_ab + dy_ap * dy_ab) / mag_ab  # AP*AB/|AB|
        return progress, side_val

    def _drive_fn(self, u_ent, v_ent):
        # --- EXTRACT COORDS ---
        if u_ent[0] == "node":
            u_coords = self.start if u_ent[1] == "start" else self.ref
            prog_u, side_u_val = self.node_cache.get(u_ent[1], (0, 0))
        else:
            u_coords = u_ent[1]["coords"]
            prog_u, side_u_val = u_ent[1].get("_cached_topo", (0, 0))

        if v_ent[0] == "node":
            v_coords = self.ref if v_ent[1] == "ref" else self.start
            prog_v, side_v_val = self.node_cache.get(v_ent[1], (0, 0))
        else:
            v_coords = v_ent[1]["coords"]
            prog_v, side_v_val = v_ent[1].get("_cached_topo", (0, 0))

        cache_key = str((tuple(u_coords), tuple(v_coords)))

        # ======================================================
        # 1️⃣ GET OR COMPUTE BASE TRAVEL TIME (NO TURNS)
        # ======================================================
        route_data = None

        if cache_key in self.distance_cache:
            base_time = self.distance_cache[cache_key]
        else:
            route_data = routing.get_route(u_coords, v_coords, custom_G=self.graph)

            if route_data["travel_time"] == float("inf"):
                dist = geodesic(u_coords, v_coords).meters
                base_time = (dist / 8.33) + 30.0
            else:
                base_time = route_data["travel_time"]

            base_time = max(1.0, base_time)
            self.distance_cache[cache_key] = base_time
        # ======================================================
        # 3️⃣ APPLY TRAFFIC MULTIPLIER
        # ======================================================
        final_duration = base_time * self.traffic_multiplier

        # ======================================================
        # 4️⃣ GEOMETRIC PENALTIES
        # ======================================================
        side_u = "LEFT" if side_u_val > 0 else "RIGHT"
        side_v = "LEFT" if side_v_val > 0 else "RIGHT"

        geometric_penalty = 0.0

        if side_u == side_v and prog_v < prog_u - 0.00001:
            geometric_penalty = 1800.0
        elif side_u != side_v:
            geometric_penalty = 1200.0

        return final_duration + geometric_penalty

    def _walk_fn(self, spot_coords):
        return (
            geodesic(spot_coords, self.dest).meters / 1.3
        )  # 1.3 m/s = average human walking speed

    def get_state(self):
        return {
            "spots": self.spots,
            "start_node": "start",
            "dest_coords": self.dest,
            "destination_C": self.ref,
            "drive_fn": self._drive_fn,
            "walk_fn": self._walk_fn,
            "grid": type("Mock", (), {"nodes": {"start": self.start, "ref": self.ref}}),
            "topology": self.spot_topology,
            "start_progress": self.start_progress,
        }


In [118]:
import json
import os
from geopy.distance import geodesic
import math
import routing

class StateAdapter:
    def __init__(self, start, dest, ref, spots, traffic_multiplier=1.4, distance_cache=None, graph = None):
        self.start, self.dest, self.ref = tuple(start), tuple(dest), tuple(ref)
        self.spots = spots
        self.traffic_multiplier = traffic_multiplier
        # This cache now stores BASE travel time (without traffic multiplier)
        self.distance_cache = distance_cache if distance_cache is not None else {}
        self.graph = graph if graph is not None else routing.G
        # Astana Longitude Scaling (51.1N)
        self.lng_scale = math.cos(math.radians(self.start[0]))
        self.axis_b = self.ref 
        
        self.spot_topology = []
        self.node_cache = {
            'start': self._map_to_avenue(self.start),
            'ref': self._map_to_avenue(self.ref),
            'dest': self._map_to_avenue(self.dest)
        }
        
        for s in self.spots:
            progress, side_val = self._map_to_avenue(s['coords'])
            s['_cached_topo'] = (progress, side_val)
            self.spot_topology.append({
                'progress': progress,
                'side': "LEFT" if side_val > 0 else "RIGHT"
            })
            
        self.start_progress = self.node_cache['start'][0]
        print(f"[Topology] Mapped {len(self.spots)} spots. Cache loaded with {len(self.distance_cache)} routes.")

    def _map_to_avenue(self, point):
        yA, xA = self.start[0], self.start[1] * self.lng_scale
        yB, xB = self.axis_b[0], self.axis_b[1] * self.lng_scale
        yP, xP = point[0], point[1] * self.lng_scale
        
        side_val = (xP - xA) * (yB - yA) - (yP - yA) * (xB - xA)
        dx_ab, dy_ab = xB - xA, yB - yA
        dx_ap, dy_ap = xP - xA, yP - yA
        mag_ab = math.sqrt(dx_ab**2 + dy_ab**2)
        if mag_ab == 0: return 0, 0
        progress = (dx_ap * dx_ab + dy_ap * dy_ab) / mag_ab
        return progress, side_val

    def _drive_fn(self, u_ent, v_ent):
        """
        Calculates drive time with Geometric Traffic Constraints.
        """
        # --- EXTRACT COORDS ---
        if u_ent[0] == 'node':
            u_coords = self.start if u_ent[1] == 'start' else self.ref
            prog_u, side_u_val = self.node_cache.get(u_ent[1], (0, 0))
        else:
            u_coords = u_ent[1]['coords']
            prog_u, side_u_val = u_ent[1].get('_cached_topo', (0,0))

        if v_ent[0] == 'node':
            v_coords = self.ref if v_ent[1] == 'ref' else self.start
            prog_v, side_v_val = self.node_cache.get(v_ent[1], (0, 0))
        else:
            v_coords = v_ent[1]['coords']
            prog_v, side_v_val = v_ent[1].get('_cached_topo', (0,0))
        
        # --- CACHE KEY ---
        # cache_key = str((tuple(u_coords), tuple(v_coords)))
        
        # base_time = 0
        # if cache_key in self.distance_cache:
        #     base_time = self.distance_cache[cache_key]
        # else:
        #     route_data = routing.get_route(u_coords, v_coords)
        #     if route_data['travel_time'] == float('inf'):
        #         dist = geodesic(u_coords, v_coords).meters
        #         base_time = (dist / 8.33) + 30.0
        #     else:
        #         base_time = route_data['travel_time']
        #     base_time = max(1.0, base_time)
        #     self.distance_cache[cache_key] = base_time
        route_data = routing.get_route(u_coords, v_coords, self.graph)
        if route_data['travel_time'] == float('inf'):
            dist = geodesic(u_coords, v_coords).meters
            base_time = (dist / 8.33) + 30.0
        else:
            base_time = route_data['travel_time']
        base_time = max(1.0, base_time)
        final_duration = base_time * self.traffic_multiplier

        # --- ASYMMETRY ENFORCER ---
        side_u = "LEFT" if side_u_val > 0 else "RIGHT"
        side_v = "LEFT" if side_v_val > 0 else "RIGHT"
        
        penalty = 0.0
        # If moving backward on the same side (Backtracking)
        if side_u == side_v and prog_v < prog_u - 0.00001:
            penalty = 1800.0 # 30 min penalty
        # If switching sides (Crossing the avenue)
        elif side_u != side_v:
            penalty = 1200.0 # 20 min penalty
            
        return final_duration + penalty


    def _walk_fn(self, spot_coords):
        return geodesic(spot_coords, self.dest).meters / 1.3

    def get_state(self):
        return {
            'spots': self.spots, 'start_node': 'start', 
            'dest_coords': self.dest, 'destination_C': self.ref, 
            'drive_fn': self._drive_fn, 'walk_fn': self._walk_fn,
            'grid': type('Mock', (), {'nodes': {'start': self.start, 'ref': self.ref}}),
            'topology': self.spot_topology, 'start_progress': self.start_progress
        }
        
        

In [120]:
start = (51.1282, 71.4304)
end   = (51.1200, 71.4150)

adapter = StateAdapter(
    start=start,
    dest=end,      # required but not used in drive_fn here
    ref=end,       # use same point as ref for testing
    spots=spots,      # no spots needed for this test
    traffic_multiplier=1.4, graph = G_aug
)


[Topology] Mapped 1199 spots. Cache loaded with 0 routes.


In [119]:
import importlib
import routing_new
importlib.reload(routing_new)

<module 'routing_new' from '/Users/aruzan/Downloads/repositories/Parking-system-test/routing_new.py'>

In [121]:
u_ent = ("spot", {"coords": start})
v_ent = ("spot", {"coords": end})

time_seconds = adapter._drive_fn(u_ent, v_ent)

print("Drive time (seconds):", time_seconds)
print("Drive time (minutes):", time_seconds / 60)

Drive time (seconds): 255.69067403257904
Drive time (minutes): 4.261511233876317


In [34]:
spots[0]

{'id': 'osm_0',
 'coords': [51.159674, 71.4808744],
 'phi_exit_seconds': 0,
 'p_i': 0.5}

## Calculate Expected Time

In [35]:
import numpy as np

def calculate_metrics(chain, state):
    """
    Calculates the TRUE Expected Time (in seconds).
    
    UPDATED LOGIC:
    - Success Case: Arrival + Walk + Exit (No search penalty 'phi')
    - Failure Case: Arrival + Search Penalty 'phi' (Time spent looking/failing)
    """
    if not chain:
        return (999999.0, 0, 0, 0, 0)
        
    spots = state['spots'] # парковки
    drive_fn = state['drive_fn'] #drive time
    walk_fn = state['walk_fn'] # walk time to dest from pasking
    
    exit_mult = state.get('exit_multiplier', 1.0)
    
    total_expected_time = 0.0
    cumulative_fail_prob = 1.0
    current_timeline_time = 0.0
    prev_node = ('node', 'start')
    
    for i, idx in enumerate(chain):
        spot = spots[idx]
        p_i = spot.get('p_i', 0.8)
        phi = spot.get('phi_exit_seconds', 60)
        
        # 1. Drive to Spot
        drive_time = drive_fn(prev_node, ('spot', spot))
        arrival_time = current_timeline_time + drive_time
        
        # 2. Terminal Actions
        walk_time = walk_fn(spot['coords'])
        
        # 3. Exit Drive
        raw_exit_drive = drive_fn(('spot', spot), ('node', 'ref')) #how long it will take to leave the parking spot and drive to the reference point C
        drive_exit_ref = raw_exit_drive * exit_mult 
        
        # 4. Success Case: NO PHI. We found a spot!
        time_if_success = arrival_time + walk_time + drive_exit_ref
        
        # Add to expected value
        total_expected_time += cumulative_fail_prob * p_i * time_if_success
        
        # 5. Failure Case: INCLUDES PHI. We wasted time searching.
        time_if_fail = arrival_time + phi
        current_timeline_time = time_if_fail
        cumulative_fail_prob *= (1.0 - p_i)
        
        prev_node = ('spot', spot)

    # Penalty for failing the entire chain (Physical penalty estimate)
    total_expected_time += cumulative_fail_prob * (current_timeline_time + 1800)

    final_fail_prob = 1.0
    for idx in chain:
        final_fail_prob *= (1.0 - spots[idx].get('p_i', 0.8))
    confidence = 1.0 - final_fail_prob

    return (total_expected_time, 0, 0, 0, confidence)

In [36]:
np.zeros((3,2))

array([[0., 0.],
       [0., 0.],
       [0., 0.]])

## Algorithms

In [37]:
import numpy as np
from abc import ABC
from utils import calculate_metrics

class BaseAlgorithm(ABC):
    def __init__(self, state):
        self.state, self.spots = state, state['spots']
        self.drive_fn, self.walk_fn = state['drive_fn'], state['walk_fn']
        self.topology = state['topology']
        self.n = len(self.spots)
        
        # Initialize dynamic normalization bounds
        self.max_drive = 1.0
        self.max_walk = 1.0
        self.max_phi = 1.0
        
        # Matrices
        self.trans_matrix = np.zeros((self.n + 1, self.n)) # n+1 rows, n cols (cost of driving from one spot to other)
        self.static_matrix = np.zeros(self.n)  # 1 row with n vals (How bad is this spot if it succeeds?)
        
        # 1. First, find the maximums to establish denominators
        self._find_max_bounds()
        
        # 2. Build the topology using those bounds
        self._build_topology_matrix(exit_mult=state.get('exit_multiplier', 1.0))

    def _get_phi(self, idx):
        if idx < 0: return 0
        return self.spots[idx].get('phi_exit_seconds', 60)

    def _find_max_bounds(self): #finds max drive, walk, exit times
        """
        Dynamically calculates M_drive, M_walk, and M_phi 
        based on the actual input coordinates and spots.
        """
        drive_times = []
        walk_times = []
        phis = []

        for i in range(self.n):
            # Walking components
            walk_times.append(self.walk_fn(self.spots[i]['coords']))
            
            # Search components
            phis.append(self._get_phi(i))
            
            # Driving components (Start -> Spot and Spot -> Ref)
            drive_times.append(self.drive_fn(('node', 'start'), ('spot', self.spots[i])))
            drive_times.append(self.drive_fn(('spot', self.spots[i]), ('node', 'ref')))
            
            # Sample Spot -> Spot transitions for a better M_drive estimate
            # (We don't do all n^2 to stay efficient, but we take enough samples)
            if self.n > 1:
                next_idx = (i + 1) % self.n
                drive_times.append(self.drive_fn(('spot', self.spots[i]), ('spot', self.spots[next_idx])))

        self.max_drive = max(drive_times) if drive_times else 1.0
        self.max_walk = max(walk_times) if walk_times else 1.0
        self.max_phi = max(phis) if phis else 1.0
        
        # Guard against zero division
        if self.max_drive == 0: self.max_drive = 1.0
        if self.max_walk == 0: self.max_walk = 1.0
        if self.max_phi == 0: self.max_phi = 1.0

    def _build_topology_matrix(self, exit_mult=1.0):
        # 1. Static Cost: Quality metrics normalized by dynamic maximums
        for i in range(self.n):
            walk = self.walk_fn(self.spots[i]['coords'])
            raw_exit = self.drive_fn(('spot', self.spots[i]), ('node', 'ref'))
            # Static Success Cost = Normalized Walk + Normalized Exit
            self.static_matrix[i] = (walk / self.max_walk) + ((raw_exit * exit_mult) / self.max_drive) #how bad is this spot compared to all other spots
            
        # 2. Transition Matrix: Real road times
        for u in range(-1, self.n):
            u_prog = self.state['start_progress'] if u == -1 else self.topology[u]['progress']
            u_node = ('node', 'start') if u == -1 else ('spot', self.spots[u])
            
            for v in range(self.n):
                if u == v: 
                    self.trans_matrix[u+1, v] = float('inf')
                    continue
                
                v_prog = self.topology[v]['progress']
                # Zero-tolerance backtracking
                if v_prog < u_prog: 
                    self.trans_matrix[u+1, v] = float('inf')
                    continue
                
                self.trans_matrix[u+1, v] = self.drive_fn(u_node, ('spot', self.spots[v]))


#Given many possible parking spots, each with probability 
# of success and costs, find an ordered chain of at most k spots 
# that minimizes the expected total time of the trip.

class MDP_Difference(BaseAlgorithm):
    def solve(self, **kwargs):
        k, lw, le, ltr = int(kwargs.get('k', 3)), kwargs.get('lambda_w', 1.0), kwargs.get('lambda_e', 1.0), kwargs.get('lambda_tr', 1.0)
        beam = [(0.0, 1.0, -1, [], frozenset())]; best_chain = [] #(cost_so_far, fail_probability, current_spot, path_taken, visited_spots)
        
        for _ in range(k):
            candidates = []
            for cost_so_far, fail_prob, curr, path, visited in beam:
                # Search penalty of failed spot normalized by M_phi
                phi_prev = self._get_phi(curr) if curr >= 0 else 0
                norm_phi_prev = phi_prev / self.max_phi
                
                for i in range(self.n):
                    raw_drive = self.trans_matrix[curr + 1, i] #driving time from current position to spot i
                    if raw_drive == float('inf') or i in visited: continue
                    
                    p_i = self.spots[i].get('p_i', 0.8)
                    walk = self.walk_fn(self.spots[i]['coords'])
                    exit_d = self.drive_fn(('spot', self.spots[i]), ('node', 'ref'))
                    
                    # Transition: Phi_prev + Drive(u, v)
                    drive_w = ltr if curr >= 0 else 1.0
                    norm_drive = (drive_w * raw_drive) / self.max_drive
                    norm_step = norm_drive + norm_phi_prev
                    
                    # Quality: Normalized by respective M values
                    norm_quality = (lw * (walk / self.max_walk)) + (le * (exit_d / self.max_drive))
                    
                    q = norm_step + norm_quality
                    candidates.append((cost_so_far + fail_prob * q, fail_prob * (1.0 - p_i), i, path + [i], visited | {i}))
            
            candidates.sort(key=lambda x: x[0])
            beam = candidates[:1000]
            if beam: best_chain = beam[0][3]
        return best_chain, calculate_metrics(best_chain, self.state), {}

class FiniteHorizonMDP(BaseAlgorithm):
    def solve(self, **kwargs):
        k, lw, le, ltr = int(kwargs.get('k', 3)), kwargs.get('lambda_w', 1.0), kwargs.get('lambda_e', 1.0), kwargs.get('lambda_tr', 1.0)
        beam = [(0.0, 1.0, -1, [], frozenset())]; best_completed = None; min_cost = float('inf')
        
        for step in range(max(k, 7)):
            candidates = []
            for cost, fail_p, curr, path, visited in beam:
                if fail_p < 0.005:
                    if cost < min_cost: min_cost, best_completed = cost, path
                    continue
                phi_prev = self._get_phi(curr) if curr >= 0 else 0
                for i in range(self.n):
                    raw_drive = self.trans_matrix[curr + 1, i]
                    if raw_drive == float('inf') or i in visited: continue
                    p_i, walk = self.spots[i].get('p_i', 0.8), self.walk_fn(self.spots[i]['coords'])
                    exit_d, phi_i = self.drive_fn(('spot', self.spots[i]), ('node', 'ref')), self._get_phi(i)
                    drive_w = ltr if curr >= 0 else 1.0
                    n_drive = (drive_w * raw_drive) / self.max_drive
                    n_phi_p = phi_prev / self.max_phi
                    
                    c_success = (n_drive + n_phi_p) + (lw * walk / self.max_walk) + (le * exit_d / self.max_drive)
                    c_fail = (n_drive + n_phi_p) + (phi_i / self.max_phi)
                    
                    candidates.append((cost + fail_p * (p_i * c_success + (1.0 - p_i) * c_fail), fail_p * (1.0 - p_i), i, path + [i], visited | {i}))
            candidates.sort(key=lambda x: x[0]); beam = candidates[:1000]
            if beam and (1.0 - beam[0][1]) >= 0.9 and beam[0][0] < min_cost:
                min_cost, best_completed = beam[0][0], beam[0][3]
        return (best_completed or beam[0][3]), calculate_metrics(best_completed or beam[0][3], self.state), {}

class HeuristicLookahead(BaseAlgorithm):
    def solve(self, **kwargs):
        k, lw, le, ltr = int(kwargs.get('k', 3)), kwargs.get('lambda_w', 1.0), kwargs.get('lambda_e', 1.0), kwargs.get('lambda_tr', 1.0)
        chain, curr = [], -1
        for _ in range(k):
            best_i, min_q = -1, float('inf')
            phi_prev = self._get_phi(curr) if curr >= 0 else 0
            for i in range(self.n):
                raw_drive = self.trans_matrix[curr + 1, i]
                if raw_drive == float('inf') or i in chain: continue
                p_i = self.spots[i].get('p_i', 0.8)
                walk, exit_d = self.walk_fn(self.spots[i]['coords']), self.drive_fn(('spot', self.spots[i]), ('node', 'ref'))
                drive_w = ltr if curr >= 0 else 1.0
                n_step = ((drive_w * raw_drive) / self.max_drive) + (phi_prev / self.max_phi)
                n_quality = (lw * walk / self.max_walk) + (le * exit_d / self.max_drive) + (self._get_phi(i) / self.max_phi)
                q = n_step + (n_quality / max(p_i, 0.1))
                if q < min_q: min_q, best_i = q, i
            if best_i == -1: break
            chain.append(best_i); curr = best_i
        return chain, calculate_metrics(chain, self.state), {}

In [50]:
['a', 'b'] + [1]

['a', 'b', 1]

In [129]:
import routing
import osmnx as ox

# 1️⃣ Инициализация
routing.initialize_graph()

u = (51.1694, 71.4491)
v = (51.1700, 71.4600)

# 2️⃣ Маршрут с поворотами
res_with_turns = routing.get_route(u, v)

# 3️⃣ Отключаем повороты
old_vals = (
    routing.COST_LEFT,
    routing.COST_RIGHT,
    routing.COST_U_TURN,
    routing.COST_STRAIGHT,
)

routing.COST_LEFT = 0
routing.COST_RIGHT = 0
routing.COST_U_TURN = 0
routing.COST_STRAIGHT = 0

res_no_turns = routing.get_route(u, v)

# Вернуть штрафы
(
    routing.COST_LEFT,
    routing.COST_RIGHT,
    routing.COST_U_TURN,
    routing.COST_STRAIGHT,
) = old_vals

# 4️⃣ Вывести время
print("Time without turns:", res_no_turns["travel_time"])
print("Time with turns:", res_with_turns["travel_time"])



Time without turns: 82.4868678292864
Time with turns: 202.48686782928638


In [130]:
import folium
import routing

routing.initialize_graph()

u = (51.1694, 71.4491)
v = (51.1700, 71.4600)

res = routing.get_route(u, v)

# Центр карты
center = u
m = folium.Map(location=center, zoom_start=14)

# Линия маршрута
folium.PolyLine(res["path"], weight=5).add_to(m)

# Маркеры
folium.Marker(u, tooltip="Start").add_to(m)
folium.Marker(v, tooltip="End").add_to(m)

m.save("route.html")


In [131]:
import folium
import routing

routing.initialize_graph()

u = (51.1694, 71.4491)
v = (51.1700, 71.4600)

res = routing.get_route(u, v)

m = folium.Map(location=u, zoom_start=14)

# Маршрут
folium.PolyLine(res["path"], weight=5, color="blue").add_to(m)

# Start — зелёный
folium.Marker(
    u,
    tooltip="Start",
    icon=folium.Icon(color="green")
).add_to(m)

# End — красный
folium.Marker(
    v,
    tooltip="End",
    icon=folium.Icon(color="red")
).add_to(m)

m.save("route.html")


In [6]:
import networkx as nx  # Graph logic: routing, shortest paths on road networks
import osmnx as ox  # Download real road networks from OpenStreetMap
import os  # File and path handling (check/save/load files)
import pickle  # Save/load Python objects (cache graphs, data)
import math  # Geometry and numeric calculations (angles, distances)
from functools import lru_cache  # Cache function results to avoid recomputation
from shapely.geometry import Point, LineString  # Represent locations as map points
from shapely.ops import nearest_points  # Find closest geometry (e.g., spot → road)

# ==========================================
# CONFIGURATION
# ==========================================
PLACE_NAME = "Astana, Kazakhstan"
GRAPH_FILENAME = "astana_drive.graphml"

# PENALTIES (Seconds)
COST_U_TURN = 20.0
COST_LEFT = 10.0
COST_RIGHT = 5.0
COST_STRAIGHT = 0.0
# COST_U_TURN = 0.0
# COST_LEFT = 0.0
# COST_RIGHT = 0.0
# COST_STRAIGHT = 0.0

G = None
G_TURN = None

BUSY_STREETS_1 = ["Мәңгілік Ел даңғылы", "Тұран даңғылы", "Қабанбай Батыр даңғылы"]

BUSY_STREETS_2 = [
    "Сарайшық көшесі",
    "Қайым Мұхамедханов көшесі",
    "Мухамедханова",
    "Сығанақ көшесі",
    "проспект Республики",
]

BUSY_STREETS_3 = [
    "улица Шокана Валиханова",
    "Тәуелсіздік даңғылы",
    "Арай көшесі",
    "Шәкәрім Құдайбердіұлы даңғылы",
]

HARD_INTERSECTIONS = [
    ("Тұран даңғылы", "Коргалжын шоссе"),
    ("Достық көшесі", "Ақмешіт көшесі"),
    ("Тұран даңғылы", "Сарайшық көшесі"),
    ("Достық көшесі", "Түркістан көшесі"),
    ("Сарайшық көшесі", "Ақмешіт көшесі"),
    ("Қабанбай Батыр даңғылы", "Сарайшық көшесі"),
    ("Қабанбай Батыр даңғылы", "Қонаев көшесі"),
    ("Қабанбай Батыр даңғылы", "Достық көшесі"),
    ("Достық көшесі", "Сауран көшесі"),
    ("Қонаев көшесі", "Түркістан көшесі"),
    ("Қабанбай Батыр даңғылы", "Коргалжын шоссе"),
    ("Тәуелсіздік даңғылы", "Бауыржан Момышұлы даңғылы"),
    ("улица Кенесары", "проспект Женис"),
    ("проспект Республики", "улица Сейфуллина"),
    ("проспект Республики", "улица Сакена Сейфуллина"),
    ("улица Шокана Валиханова", "улица Сейфуллина"),
    ("улица Шокана Валиханова", "улица Сакена Сейфуллина"),
]


def initialize_graph():  # запуск карты
    global G, G_TURN
    if G is not None:
        return  # Если карта уже есть -> ничего не делаем.
    if os.path.exists(GRAPH_FILENAME):
        G = ox.load_graphml(GRAPH_FILENAME)
    else:
        G = ox.graph_from_place(PLACE_NAME, network_type="drive")
        ox.save_graphml(G, GRAPH_FILENAME)

    # Add travel times if missing
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)
    G = ox.add_edge_bearings(G)
    # ---- Add travel_time_new ----
    for u, v, k, data in G.edges(keys=True, data=True):
        base = data.get("travel_time", 1.0)
        name = data.get("name")

        multiplier = 1.0

        if name:
            if isinstance(name, list):
                names = name
            else:
                names = [name]

            if any(n in BUSY_STREETS_1 for n in names):
                multiplier = 1.4
            elif any(n in BUSY_STREETS_2 for n in names):
                multiplier = 1.3
            elif any(n in BUSY_STREETS_3 for n in names):
                multiplier = 1.2

        data["travel_time_new"] = base * multiplier


In [9]:
initialize_graph()

In [10]:
def check_busy_edges(G):
    for u, v, k, data in G.edges(keys=True, data=True):
        name = data.get("name")
        if not name:
            continue

        names = name if isinstance(name, list) else [name]

        if any(n in BUSY_STREETS_1 for n in names):
            print("FOUND BUSY_1:", names)
            print("base:", data.get("travel_time"))
            print("new :", data.get("travel_time_new"))
            print("ratio:", data.get("travel_time_new") / data.get("travel_time"))
            print("-" * 40)
            break

In [11]:
changed = 0
total = 0

for _, _, _, data in G.edges(keys=True, data=True):
    base = data.get("travel_time")
    new = data.get("travel_time_new")

    if base and new:
        total += 1
        if abs(new - base) > 1e-6:
            changed += 1

print("Total edges:", total)
print("Changed edges:", changed)

Total edges: 21030
Changed edges: 798


In [12]:
ALL_BUSY = set(BUSY_STREETS_1 + BUSY_STREETS_2 + BUSY_STREETS_3)

def check_busy_edges(G):
    found = 0

    for u, v, k, data in G.edges(keys=True, data=True):
        name = data.get("name")
        if not name:
            continue

        names = name if isinstance(name, list) else [name]

        for n in names:
            if n in ALL_BUSY:
                base = data.get("travel_time")
                new = data.get("travel_time_new")

                print("Street:", n)
                print("travel_time:", base)
                print("travel_time_new:", new)

                if base and new:
                    print("ratio:", round(new / base, 3))

                print("-" * 40)
                found += 1
                break

    print("Total matched edges:", found)

In [13]:
check_busy_edges(G)

Street: Қабанбай Батыр даңғылы
travel_time: 11.533820626851742
travel_time_new: 16.14734887759244
ratio: 1.4
----------------------------------------
Street: Сарайшық көшесі
travel_time: 0.40778580184937546
travel_time_new: 0.5301215424041881
ratio: 1.3
----------------------------------------
Street: Қабанбай Батыр даңғылы
travel_time: 15.208189624341673
travel_time_new: 21.29146547407834
ratio: 1.4
----------------------------------------
Street: Қабанбай Батыр даңғылы
travel_time: 0.9277218988014514
travel_time_new: 1.2988106583220318
ratio: 1.4
----------------------------------------
Street: Қабанбай Батыр даңғылы
travel_time: 17.80545411561972
travel_time_new: 24.92763576186761
ratio: 1.4
----------------------------------------
Street: Қабанбай Батыр даңғылы
travel_time: 0.45760931913340513
travel_time_new: 0.6406530467867672
ratio: 1.4
----------------------------------------
Street: проспект Республики
travel_time: 15.781087393719794
travel_time_new: 20.515413611835733
ratio: 

In [14]:
def augment_graph_with_spots(spots):
    """
    Realizes the 'Split-Edge' theory.
    Transforms the graph by injecting spot-nodes into existing edges.
    """
    global G
    initialize_graph()  # убеждаемся, что карта дорог загружена

    # Work on a copy to keep the base graph clean
    G_aug = G.copy()

    # 1. Find nearest edges for all spots
    lats = [s["coords"][0] for s in spots]
    lngs = [s["coords"][1] for s in spots]
    nearest_edges = ox.nearest_edges(
        G_aug, lngs, lats
    )  # На какой именно улице она лежит ближе всего?

    # 2. Group spots by the edge they belong to
    edge_to_spots = {}
    for i, edge in enumerate(nearest_edges):
        ######### CHANGE ##########
        spots[i]["_edge"] = edge
        if edge not in edge_to_spots:
            edge_to_spots[edge] = []
        edge_to_spots[edge].append(spots[i])

    for (u, v, k), spot_list in edge_to_spots.items():
        edge_data = G_aug.get_edge_data(u, v, k)
        if not edge_data:
            continue
        ########### CHANGE ################
        bearing = edge_data.get("bearing", 0)

        edge_geom = edge_data.get("geometry", None)  ### we have None values too
        total_len = edge_data["length"]
        total_time = edge_data["travel_time_new"]  # CHANGE from travel_time

        # Calculate distance along edge for each spot
        spot_offsets = []
        for s in spot_list:
            p = Point(s["coords"][1], s["coords"][0])
            ########### CHANGE ##########
            s["_bearing"] = bearing
            ##### CHANGE ###############
            if edge_geom is None:
                # build straight-line geometry from u to v
                u_x, u_y = G_aug.nodes[u]["x"], G_aug.nodes[u]["y"]
                v_x, v_y = G_aug.nodes[v]["x"], G_aug.nodes[v]["y"]
                edge_geom = LineString([(u_x, u_y), (v_x, v_y)])
            dist_along = edge_geom.project(p)
            ##### CHANGE ###############
            spot_offsets.append((s["id"], dist_along, s["coords"]))

        # Sort spots by distance along the edge
        spot_offsets.sort(key=lambda x: x[1])

        # 3. Split the edge
        curr_u = u
        last_offset = 0

        for spot_id, offset, coords in spot_offsets:
            new_node = f"spot_{spot_id}"
            ###### CHANGE 1 ##########
            proj_point = edge_geom.interpolate(offset)

            G_aug.add_node(
                new_node,
                x=proj_point.x,
                y=proj_point.y,
            )
            ###### CHANGE 1 ##########

            # Calculate segment metrics
            seg_len = offset - last_offset
            ratio = seg_len / total_len if total_len > 0 else 0
            seg_time = total_time * ratio

            # Add the segment from current position to this spot
            G_aug.add_edge(
                curr_u,
                new_node,
                length=seg_len,
                travel_time_new=seg_time,  # CHANGE from travel_time
                highway=edge_data.get("highway"),
            )

            curr_u = new_node
            last_offset = offset

        # Final segment to the original end intersection
        final_len = total_len - last_offset
        ratio = final_len / total_len if total_len > 0 else 0
        G_aug.add_edge(
            curr_u,
            v,
            length=final_len,
            travel_time_new=total_time * ratio,  # CHANGE from travel_time
            highway=edge_data.get("highway"),
        )

        # Remove the original long edge
        G_aug.remove_edge(u, v, k)

    return G_aug

In [19]:
import json

with open("manual_spots_new.json", "r", encoding="utf-8") as f:
    spots = json.load(f)

print(type(spots))        # should be list
print(spots[0])           # should be dict with "id" and "coords"

<class 'list'>
{'id': 'manual_1768384179610', 'coords': [51.12041474774691, 71.42925381660463], 'street': 'Manual Entry', 'type': 'surface', 'is_manual': True, 'phi_exit_seconds': 0, 'p_i': 0.5, '_cached_topo': [0.0045344476166785915, -3.209893679861463e-06]}


In [21]:
G_aug = augment_graph_with_spots(spots)

print("Original edges:", len(G.edges))
print("Augmented edges:", len(G_aug.edges))
print("Original nodes:", len(G.nodes))
print("Augmented nodes:", len(G_aug.nodes))

Original edges: 21030
Augmented edges: 21092
Original nodes: 8469
Augmented nodes: 8531


In [35]:
BUSY_STREETS_1 = ["Мәңгілік Ел даңғылы", "Тұран даңғылы", "Қабанбай Батыр даңғылы"]

BUSY_STREETS_2 = [
    "Сарайшық көшесі",
    "Қайым Мұхамедханов көшесі",
    "Мухамедханова",
    "Сығанақ көшесі",
    "проспект Республики",
]

BUSY_STREETS_3 = [
    "улица Шокана Валиханова",
    "Тәуелсіздік даңғылы",
    "Арай көшесі",
    "Шәкәрім Құдайбердіұлы даңғылы",
]

HARD_INTERSECTIONS = [
    ("Тұран даңғылы", "Коргалжын шоссе"),
    ("Достық көшесі", "Ақмешіт көшесі"),
    ("Тұран даңғылы", "Сарайшық көшесі"),
    ("Достық көшесі", "Түркістан көшесі"),
    ("Сарайшық көшесі", "Ақмешіт көшесі"),
    ("Қабанбай Батыр даңғылы", "Сарайшық көшесі"),
    ("Қабанбай Батыр даңғылы", "Қонаев көшесі"),
    ("Қабанбай Батыр даңғылы", "Достық көшесі"),
    ("Достық көшесі", "Сауран көшесі"),
    ("Қонаев көшесі", "Түркістан көшесі"),
    ("Қабанбай Батыр даңғылы", "Коргалжын шоссе"),
    ("Тәуелсіздік даңғылы", "Бауыржан Момышұлы даңғылы"),
    ("улица Кенесары", "проспект Женис"),
    ("проспект Республики", "улица Сейфуллина"),
    ("проспект Республики", "улица Сакена Сейфуллина"),
    ("улица Шокана Валиханова", "улица Сейфуллина"),
    ("улица Шокана Валиханова", "улица Сакена Сейфуллина"),
]

In [43]:
def get_route(u_coords, v_coords, custom_G=None, use_penalties=True):
    target_G = custom_G if custom_G else G

    COST_U_TURN = 20.0
    COST_LEFT = 10.0
    COST_RIGHT = 5.0
    COST_STRAIGHT = 0.0

    u_node = (
        u_coords
        if isinstance(u_coords, str)
        else ox.nearest_nodes(target_G, u_coords[1], u_coords[0])
    )
    v_node = (
        v_coords
        if isinstance(v_coords, str)
        else ox.nearest_nodes(target_G, v_coords[1], v_coords[0])
    )

    try:
        path = nx.shortest_path(target_G, u_node, v_node, weight="travel_time_new")
        path_coords = [[target_G.nodes[n]["y"], target_G.nodes[n]["x"]] for n in path]

        # -------------------------
        # BASE EDGE TIME
        # -------------------------
        base_time = 0.0
        for i in range(len(path) - 1):
            edge_data = target_G.get_edge_data(path[i], path[i + 1])
            if isinstance(edge_data, dict):
                edge_data = list(edge_data.values())[0]
            base_time += edge_data.get("travel_time_new", 1.0)

        # -------------------------
        # PENALTIES
        # -------------------------
        turn_penalty = 0.0

        counts = {
            "left": 0,
            "right": 0,
            "uturn": 0,
            "straight": 0,
            "signals": 0,
            "hard_intersections": 0,
        }

        if use_penalties:
            path_without_spots = [n for n in path if not isinstance(n, str)]

            for i in range(1, len(path_without_spots) - 1):
                n1, n2, n3 = (
                    path_without_spots[i - 1],
                    path_without_spots[i],
                    path_without_spots[i + 1],
                )

                edge1 = target_G.get_edge_data(n1, n2)
                edge2 = target_G.get_edge_data(n2, n3)

                if not edge1 or not edge2:
                    continue

                if isinstance(edge1, dict):
                    edge1 = list(edge1.values())[0]
                if isinstance(edge2, dict):
                    edge2 = list(edge2.values())[0]

                b1 = edge1.get("bearing", 0)
                b2 = edge2.get("bearing", 0)
                angle = (b2 - b1 + 180) % 360 - 180

                if abs(angle) < 20:
                    turn_type = "straight"
                    base_turn = COST_STRAIGHT
                elif abs(angle) > 150:
                    turn_type = "uturn"
                    base_turn = COST_U_TURN
                elif angle > 0:
                    turn_type = "right"
                    base_turn = COST_RIGHT
                else:
                    turn_type = "left"
                    base_turn = COST_LEFT

                counts[turn_type] += 1

                # --- Hard intersection ---
                names1 = edge1.get("name")
                names2 = edge2.get("name")

                is_hard = False
                if names1 and names2:
                    if not isinstance(names1, list):
                        names1 = [names1]
                    if not isinstance(names2, list):
                        names2 = [names2]

                    is_hard = any(
                        (a, b) in HARD_INTERSECTIONS or (b, a) in HARD_INTERSECTIONS
                        for a in names1
                        for b in names2
                    )

                if is_hard:
                    counts["hard_intersections"] += 1

                # --- Traffic signal ---
                has_signal = target_G.nodes[n2].get("highway") == "traffic_signals"

                signal_penalty = 0
                if has_signal:
                    counts["signals"] += 1

                    if turn_type == "right":
                        signal_penalty = 5
                    elif turn_type == "straight":
                        signal_penalty = 20
                    elif turn_type == "left":
                        signal_penalty = 30
                    elif turn_type == "uturn":
                        signal_penalty = 35

                if is_hard:
                    signal_penalty *= 2.0

                total_added = base_turn + signal_penalty
                turn_penalty += total_added

                # DEBUG
                print(f"Turn at node {n2}")
                print(f"  Type: {turn_type}")
                print(f"  Angle: {round(angle,2)}")
                print(f"  Base cost: {base_turn}")
                print(f"  Signal cost: {signal_penalty}")
                print(f"  Hard intersection: {is_hard}")
                print(f"  Total added: {total_added}")
                print("-" * 40)

        return {
            "path": path_coords,
            "base_time": base_time,
            "penalty_time": turn_penalty,
            "total_time": base_time + turn_penalty,
            "counts": counts,
            "nodes": path,
        }

    except Exception as e:
        print("Routing error:", e)
        return None

In [37]:
import routing

# 1️⃣ Initialize graph
routing.initialize_graph()

u = (51.133696570341655, 71.43163913937695)
v = (51.124765685067246, 71.43210200919789)

In [45]:
res = get_route(u, v, use_penalties=True)

print("Base time:", res["base_time"])
print("Penalty time:", res["penalty_time"])
print("Total time:", res["total_time"])
print("Counts:", res["counts"])

Turn at node 6539420680
  Type: left
  Angle: -88.54
  Base cost: 10.0
  Signal cost: 0
  Hard intersection: False
  Total added: 10.0
----------------------------------------
Turn at node 6956766041
  Type: left
  Angle: -86.88
  Base cost: 10.0
  Signal cost: 0
  Hard intersection: False
  Total added: 10.0
----------------------------------------
Turn at node 260031814
  Type: straight
  Angle: -4.27
  Base cost: 0.0
  Signal cost: 0
  Hard intersection: False
  Total added: 0.0
----------------------------------------
Turn at node 6539420682
  Type: straight
  Angle: 0.26
  Base cost: 0.0
  Signal cost: 20
  Hard intersection: False
  Total added: 20.0
----------------------------------------
Turn at node 6539420681
  Type: left
  Angle: -90.3
  Base cost: 10.0
  Signal cost: 30
  Hard intersection: False
  Total added: 40.0
----------------------------------------
Turn at node 6539398090
  Type: straight
  Angle: -0.03
  Base cost: 0.0
  Signal cost: 20
  Hard intersection: False


In [39]:
res_yes = get_route(u, v, use_penalties=True)

print("Base time:", res_yes["base_time"])
print("Penalty time:", res_yes["penalty_time"])
print("Total time:", res_yes["total_time"])
print("Counts:", res_yes["counts"])

Base time: 79.32895049157213
Penalty time: 225.0
Total time: 304.32895049157213
Counts: {'left': 4, 'right': 1, 'uturn': 0, 'straight': 8, 'signals': 7, 'hard_intersections': 2}


In [46]:
import osmnx as ox
import networkx as nx
import folium
from itertools import islice

# Use your routing file if needed
import routing

routing.initialize_graph()
G = routing.G

In [47]:
RITZ_CARLTON = [51.124765685067246, 71.43210200919789]
KHAN_SHATYR = [51.132599389420065, 71.40672935111475]

In [48]:
def path_to_coords(path, G):
    return [[G.nodes[n]["y"], G.nodes[n]["x"]] for n in path]

In [49]:
def get_10_base_routes(start, end, G):
    u = ox.nearest_nodes(G, start[1], start[0])
    v = ox.nearest_nodes(G, end[1], end[0])

    gen = nx.shortest_simple_paths(G, u, v, weight="travel_time_new")
    paths = list(islice(gen, 10))

    results = []

    for path in paths:
        base_time = 0
        for i in range(len(path) - 1):
            edge = list(G.get_edge_data(path[i], path[i+1]).values())[0]
            base_time += edge.get("travel_time_new", 1.0)

        results.append({
            "path": path,
            "coords": path_to_coords(path, G),
            "base_time": base_time
        })

    return results

In [50]:
def get_5_penalized_routes(start, end, G):
    return routing.get_k_best_routes(
        start, end,
        custom_G=G,
        k_generate=10,
        k_return=5
    )

In [51]:
def draw_routes_map(start, end, routes, time_key, title):
    m = folium.Map(location=start, zoom_start=14)

    folium.Marker(start, popup="Start", icon=folium.Icon(color="green")).add_to(m)
    folium.Marker(end, popup="End", icon=folium.Icon(color="red")).add_to(m)

    colors = ["blue", "orange", "purple", "black", "darkred",
              "cadetblue", "darkgreen", "pink", "gray", "lightblue"]

    for i, r in enumerate(routes):
        folium.PolyLine(
            r["coords"] if "coords" in r else r["path"],
            color=colors[i % len(colors)],
            weight=4,
            opacity=0.8,
            tooltip=f"{i+1}: {round(r[time_key],1)} sec"
        ).add_to(m)

    display(m)

In [54]:
def to_digraph(G_multi):
    G_simple = nx.DiGraph()

    for u, v, data in G_multi.edges(data=True):
        weight = data.get("travel_time_new", 1.0)

        # keep only best parallel edge
        if G_simple.has_edge(u, v):
            if weight < G_simple[u][v]["travel_time_new"]:
                G_simple[u][v].update(data)
        else:
            G_simple.add_edge(u, v, **data)

    # copy node attributes
    for n, data in G_multi.nodes(data=True):
        G_simple.add_node(n, **data)

    return G_simple

In [55]:
G_simple = to_digraph(G)

In [58]:
# 10 by base travel_time_new
base_routes = get_10_base_routes(RITZ_CARLTON, KHAN_SHATYR, G_simple)

draw_routes_map(
    RITZ_CARLTON,
    KHAN_SHATYR,
    base_routes,
    time_key="base_time",
    title="Top 10 by Base Travel Time"
)

KeyError: 'crs'

In [53]:
# 5 by penalized time
penalized_routes = get_5_penalized_routes(RITZ_CARLTON, KHAN_SHATYR, G)

draw_routes_map(
    RITZ_CARLTON,
    KHAN_SHATYR,
    penalized_routes,
    time_key="total_time",
    title="Top 5 by Penalized Time"
)

AttributeError: module 'routing' has no attribute 'get_k_best_routes'

In [59]:
import networkx as nx  # Graph logic: routing, shortest paths on road networks
import osmnx as ox  # Download real road networks from OpenStreetMap
import os  # File and path handling (check/save/load files)
import pickle  # Save/load Python objects (cache graphs, data)
import math  # Geometry and numeric calculations (angles, distances)
from functools import lru_cache  # Cache function results to avoid recomputation
from shapely.geometry import Point, LineString  # Represent locations as map points
from shapely.ops import nearest_points  # Find closest geometry (e.g., spot → road)
from itertools import islice

# ==========================================
# CONFIGURATION
# ==========================================
PLACE_NAME = "Astana, Kazakhstan"
GRAPH_FILENAME = "astana_drive.graphml"

# PENALTIES (Seconds)
COST_U_TURN = 20.0
COST_LEFT = 10.0
COST_RIGHT = 5.0
COST_STRAIGHT = 0.0
# COST_U_TURN = 0.0
# COST_LEFT = 0.0
# COST_RIGHT = 0.0
# COST_STRAIGHT = 0.0

G = None
G_TURN = None

BUSY_STREETS_1 = ["Мәңгілік Ел даңғылы", "Тұран даңғылы", "Қабанбай Батыр даңғылы"]

BUSY_STREETS_2 = [
    "Сарайшық көшесі",
    "Қайым Мұхамедханов көшесі",
    "Мухамедханова",
    "Сығанақ көшесі",
    "проспект Республики",
]

BUSY_STREETS_3 = [
    "улица Шокана Валиханова",
    "Тәуелсіздік даңғылы",
    "Арай көшесі",
    "Шәкәрім Құдайбердіұлы даңғылы",
]

HARD_INTERSECTIONS = [
    ("Тұран даңғылы", "Коргалжын шоссе"),
    ("Достық көшесі", "Ақмешіт көшесі"),
    ("Тұран даңғылы", "Сарайшық көшесі"),
    ("Достық көшесі", "Түркістан көшесі"),
    ("Сарайшық көшесі", "Ақмешіт көшесі"),
    ("Қабанбай Батыр даңғылы", "Сарайшық көшесі"),
    ("Қабанбай Батыр даңғылы", "Қонаев көшесі"),
    ("Қабанбай Батыр даңғылы", "Достық көшесі"),
    ("Достық көшесі", "Сауран көшесі"),
    ("Қонаев көшесі", "Түркістан көшесі"),
    ("Қабанбай Батыр даңғылы", "Коргалжын шоссе"),
    ("Тәуелсіздік даңғылы", "Бауыржан Момышұлы даңғылы"),
    ("улица Кенесары", "проспект Женис"),
    ("проспект Республики", "улица Сейфуллина"),
    ("проспект Республики", "улица Сакена Сейфуллина"),
    ("улица Шокана Валиханова", "улица Сейфуллина"),
    ("улица Шокана Валиханова", "улица Сакена Сейфуллина"),
]


def initialize_graph():  # запуск карты
    global G, G_TURN
    if G is not None:
        return  # Если карта уже есть -> ничего не делаем.
    if os.path.exists(GRAPH_FILENAME):
        G = ox.load_graphml(GRAPH_FILENAME)
    else:
        G = ox.graph_from_place(PLACE_NAME, network_type="drive")
        ox.save_graphml(G, GRAPH_FILENAME)

    # Add travel times if missing
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)
    G = ox.add_edge_bearings(G)
    # ---- Add travel_time_new ----
    for u, v, k, data in G.edges(keys=True, data=True):
        base = data.get("travel_time", 1.0)
        name = data.get("name")

        multiplier = 1.0

        if name:
            if isinstance(name, list):
                names = name
            else:
                names = [name]

            if any(n in BUSY_STREETS_1 for n in names):
                multiplier = 1.4
            elif any(n in BUSY_STREETS_2 for n in names):
                multiplier = 1.3
            elif any(n in BUSY_STREETS_3 for n in names):
                multiplier = 1.2

        data["travel_time_new"] = base * multiplier


def augment_graph_with_spots(spots):
    """
    Realizes the 'Split-Edge' theory.
    Transforms the graph by injecting spot-nodes into existing edges.
    """
    global G
    initialize_graph()  # убеждаемся, что карта дорог загружена

    # Work on a copy to keep the base graph clean
    G_aug = G.copy()

    # 1. Find nearest edges for all spots
    lats = [s["coords"][0] for s in spots]
    lngs = [s["coords"][1] for s in spots]
    nearest_edges = ox.nearest_edges(
        G_aug, lngs, lats
    )  # На какой именно улице она лежит ближе всего?

    # 2. Group spots by the edge they belong to
    edge_to_spots = {}
    for i, edge in enumerate(nearest_edges):
        ######### CHANGE ##########
        spots[i]["_edge"] = edge
        if edge not in edge_to_spots:
            edge_to_spots[edge] = []
        edge_to_spots[edge].append(spots[i])

    for (u, v, k), spot_list in edge_to_spots.items():
        edge_data = G_aug.get_edge_data(u, v, k)
        if not edge_data:
            continue
        ########### CHANGE ################
        bearing = edge_data.get("bearing", 0)

        edge_geom = edge_data.get("geometry", None)  ### we have None values too
        total_len = edge_data["length"]
        total_time = edge_data["travel_time_new"]  # CHANGE from travel_time

        # Calculate distance along edge for each spot
        spot_offsets = []
        for s in spot_list:
            p = Point(s["coords"][1], s["coords"][0])
            ########### CHANGE ##########
            s["_bearing"] = bearing
            ##### CHANGE ###############
            if edge_geom is None:
                # build straight-line geometry from u to v
                u_x, u_y = G_aug.nodes[u]["x"], G_aug.nodes[u]["y"]
                v_x, v_y = G_aug.nodes[v]["x"], G_aug.nodes[v]["y"]
                edge_geom = LineString([(u_x, u_y), (v_x, v_y)])
            dist_along = edge_geom.project(p)
            ##### CHANGE ###############
            spot_offsets.append((s["id"], dist_along, s["coords"]))

        # Sort spots by distance along the edge
        spot_offsets.sort(key=lambda x: x[1])

        # 3. Split the edge
        curr_u = u
        last_offset = 0

        for spot_id, offset, coords in spot_offsets:
            new_node = f"spot_{spot_id}"
            ###### CHANGE 1 ##########
            proj_point = edge_geom.interpolate(offset)

            G_aug.add_node(
                new_node,
                x=proj_point.x,
                y=proj_point.y,
            )
            ###### CHANGE 1 ##########

            # Calculate segment metrics
            seg_len = offset - last_offset
            ratio = seg_len / total_len if total_len > 0 else 0
            seg_time = total_time * ratio

            # Add the segment from current position to this spot
            G_aug.add_edge(
                curr_u,
                new_node,
                length=seg_len,
                travel_time_new=seg_time,  # CHANGE from travel_time
                highway=edge_data.get("highway"),
            )

            curr_u = new_node
            last_offset = offset

        # Final segment to the original end intersection
        final_len = total_len - last_offset
        ratio = final_len / total_len if total_len > 0 else 0
        G_aug.add_edge(
            curr_u,
            v,
            length=final_len,
            travel_time_new=total_time * ratio,  # CHANGE from travel_time
            highway=edge_data.get("highway"),
        )

        # Remove the original long edge
        G_aug.remove_edge(u, v, k)

    return G_aug


# def get_route(u_coords, v_coords, custom_G=None):
#     target_G = custom_G if custom_G else G

#     # --- Resolve u ---
#     if isinstance(u_coords, str):
#         u_node = u_coords
#     else:
#         u_node = ox.nearest_nodes(target_G, u_coords[1], u_coords[0])

#     # --- Resolve v ---
#     if isinstance(v_coords, str):
#         v_node = v_coords
#     else:
#         v_node = ox.nearest_nodes(target_G, v_coords[1], v_coords[0])

#     if isinstance(u_node, str) and u_node not in target_G.nodes:
#         raise KeyError(f"u_node {u_node} not found in graph")

#     if isinstance(v_node, str) and v_node not in target_G.nodes:
#         raise KeyError(f"v_node {v_node} not found in graph")

#     try:
#         path = nx.shortest_path(
#             target_G, u_node, v_node, weight="travel_time_new"
#         )  ##CHANGED from travel_time

#         path_coords = [[target_G.nodes[n]["y"], target_G.nodes[n]["x"]] for n in path]

#         time = 0.0
#         for i in range(len(path) - 1):
#             edge_data = target_G.get_edge_data(path[i], path[i + 1])
#             if isinstance(edge_data, dict):
#                 edge_data = list(edge_data.values())[0]
#             time += edge_data.get("travel_time_new", 1.0)  ##CHANGED from travel_time

#         # --- Turn penalties ---
#         turn_penalty = 0.0
#         path_without_spots = [n for n in path if not isinstance(n, str)]
#         for i in range(1, len(path_without_spots) - 1):
#             n1, n2, n3 = (
#                 path_without_spots[i - 1],
#                 path_without_spots[i],
#                 path_without_spots[i + 1],
#             )

#             edge1 = G.get_edge_data(n1, n2)
#             edge2 = G.get_edge_data(n2, n3)

#             if edge1 is None or edge2 is None:
#                 continue

#             if isinstance(edge1, dict):
#                 edge1 = list(edge1.values())[0]
#             if isinstance(edge2, dict):
#                 edge2 = list(edge2.values())[0]

#             b1 = edge1.get("bearing", 0)
#             b2 = edge2.get("bearing", 0)
#             angle = (b2 - b1 + 180) % 360 - 180
#             if abs(angle) < 20:
#                 turn_penalty += COST_STRAIGHT
#             elif abs(angle) > 150:
#                 turn_penalty += COST_U_TURN
#             elif angle > 0:
#                 turn_penalty += COST_RIGHT
#             else:
#                 turn_penalty += COST_LEFT

#         return {
#             "path": path_coords,
#             "travel_time": time + turn_penalty,
#             "nodes": path,
#         }  ### added nodes key and value
#     except Exception:
#         return {"path": [], "travel_time": float("inf"), "nodes": []}


def get_route(u_coords, v_coords, custom_G=None):
    target_G = custom_G if custom_G else G

    # --- Resolve nodes ---
    u_node = (
        u_coords
        if isinstance(u_coords, str)
        else ox.nearest_nodes(target_G, u_coords[1], u_coords[0])
    )
    v_node = (
        v_coords
        if isinstance(v_coords, str)
        else ox.nearest_nodes(target_G, v_coords[1], v_coords[0])
    )

    if isinstance(u_node, str) and u_node not in target_G.nodes:
        raise KeyError(f"u_node {u_node} not found")
    if isinstance(v_node, str) and v_node not in target_G.nodes:
        raise KeyError(f"v_node {v_node} not found")

    try:
        path = nx.shortest_path(target_G, u_node, v_node, weight="travel_time_new")

        path_coords = [[target_G.nodes[n]["y"], target_G.nodes[n]["x"]] for n in path]

        # --- Edge travel time ---
        time = 0.0
        for i in range(len(path) - 1):
            edge_data = target_G.get_edge_data(path[i], path[i + 1])
            if isinstance(edge_data, dict):
                edge_data = list(edge_data.values())[0]
            time += edge_data.get("travel_time_new", 1.0)

        # --- Turn + Intersection penalties ---
        turn_penalty = 0.0
        path_without_spots = [n for n in path if not isinstance(n, str)]

        for i in range(1, len(path_without_spots) - 1):
            n1, n2, n3 = (
                path_without_spots[i - 1],
                path_without_spots[i],
                path_without_spots[i + 1],
            )

            edge1 = G.get_edge_data(n1, n2)
            edge2 = G.get_edge_data(n2, n3)

            if not edge1 or not edge2:
                continue

            if isinstance(edge1, dict):
                edge1 = list(edge1.values())[0]
            if isinstance(edge2, dict):
                edge2 = list(edge2.values())[0]

            # ---- Turn type ----
            b1 = edge1.get("bearing", 0)
            b2 = edge2.get("bearing", 0)
            angle = (b2 - b1 + 180) % 360 - 180

            if abs(angle) < 20:
                turn_type = "straight"
                base_turn = COST_STRAIGHT
            elif abs(angle) > 150:
                turn_type = "uturn"
                base_turn = COST_U_TURN
            elif angle > 0:
                turn_type = "right"
                base_turn = COST_RIGHT
            else:
                turn_type = "left"
                base_turn = COST_LEFT

            # ---- Detect hard intersection ----
            names1 = edge1.get("name")
            names2 = edge2.get("name")

            if not names1 or not names2:
                is_hard = False
            else:
                if not isinstance(names1, list):
                    names1 = [names1]
                if not isinstance(names2, list):
                    names2 = [names2]

                is_hard = any(
                    (a, b) in HARD_INTERSECTIONS or (b, a) in HARD_INTERSECTIONS
                    for a in names1
                    for b in names2
                )

            # ---- Detect traffic signal ----
            has_signal = G.nodes[n2].get("highway") == "traffic_signals"

            # ---- Signal penalty depends on turn type ----
            signal_penalty = 0
            if has_signal:
                if turn_type == "right":
                    signal_penalty = 5
                elif turn_type == "straight":
                    signal_penalty = 20
                elif turn_type == "left":
                    signal_penalty = 30
                elif turn_type == "uturn":
                    signal_penalty = 35

            # ---- Hard intersection amplifies delay ----
            if is_hard:
                signal_penalty *= 2.0

            turn_penalty += base_turn + signal_penalty

        return {
            "path": path_coords,
            "travel_time": time + turn_penalty,
            "nodes": path,
        }

    except Exception:
        return {"path": [], "travel_time": float("inf"), "nodes": []}


def evaluate_path_with_penalties(path, target_G):
    base_time = 0.0

    for i in range(len(path) - 1):
        edge_data = target_G.get_edge_data(path[i], path[i + 1])
        if isinstance(edge_data, dict):
            edge_data = list(edge_data.values())[0]
        base_time += edge_data.get("travel_time_new", 1.0)

    turn_penalty = 0.0
    path_clean = [n for n in path if not isinstance(n, str)]

    for i in range(1, len(path_clean) - 1):
        n1, n2, n3 = path_clean[i - 1], path_clean[i], path_clean[i + 1]

        edge1 = target_G.get_edge_data(n1, n2)
        edge2 = target_G.get_edge_data(n2, n3)

        if not edge1 or not edge2:
            continue

        if isinstance(edge1, dict):
            edge1 = list(edge1.values())[0]
        if isinstance(edge2, dict):
            edge2 = list(edge2.values())[0]

        b1 = edge1.get("bearing", 0)
        b2 = edge2.get("bearing", 0)
        angle = (b2 - b1 + 180) % 360 - 180

        if abs(angle) < 20:
            turn_type = "straight"
            base_turn = COST_STRAIGHT
        elif abs(angle) > 150:
            turn_type = "uturn"
            base_turn = COST_U_TURN
        elif angle > 0:
            turn_type = "right"
            base_turn = COST_RIGHT
        else:
            turn_type = "left"
            base_turn = COST_LEFT

        has_signal = target_G.nodes[n2].get("highway") == "traffic_signals"

        signal_penalty = 0
        if has_signal:
            if turn_type == "right":
                signal_penalty = 5
            elif turn_type == "straight":
                signal_penalty = 20
            elif turn_type == "left":
                signal_penalty = 30
            elif turn_type == "uturn":
                signal_penalty = 35

        turn_penalty += base_turn + signal_penalty

    return {
        "nodes": path,
        "path": [[target_G.nodes[n]["y"], target_G.nodes[n]["x"]] for n in path],
        "base_time": base_time,
        "penalty_time": turn_penalty,
        "total_time": base_time + turn_penalty,
    }


def get_k_best_routes(u_coords, v_coords, custom_G=None, k_generate=10, k_return=5):

    target_G = custom_G if custom_G else G

    u_node = (
        u_coords
        if isinstance(u_coords, str)
        else ox.nearest_nodes(target_G, u_coords[1], u_coords[0])
    )

    v_node = (
        v_coords
        if isinstance(v_coords, str)
        else ox.nearest_nodes(target_G, v_coords[1], v_coords[0])
    )

    try:
        generator = nx.shortest_simple_paths(
            target_G, u_node, v_node, weight="travel_time_new"
        )

        candidates = list(islice(generator, k_generate))

        results = []

        for path in candidates:
            res = evaluate_path_with_penalties(path, target_G)
            results.append(res)

        results.sort(key=lambda x: x["total_time"])

        return results[:k_return]

    except:
        return []


In [61]:
initialize_graph()

In [65]:
def to_digraph(G_multi):
    G_simple = nx.DiGraph()

    # copy graph-level metadata (VERY IMPORTANT)
    G_simple.graph = G_multi.graph.copy()

    for u, v, data in G_multi.edges(data=True):
        weight = data.get("travel_time_new", 1.0)

        if G_simple.has_edge(u, v):
            if weight < G_simple[u][v]["travel_time_new"]:
                G_simple[u][v].update(data)
        else:
            G_simple.add_edge(u, v, **data)

    for n, data in G_multi.nodes(data=True):
        G_simple.add_node(n, **data)

    return G_simple

G_simple = to_digraph(G)

In [66]:
RITZ_CARLTON = [51.124765685067246, 71.43210200919789]
KHAN_SHATYR = [51.132599389420065, 71.40672935111475]

In [67]:
def get_10_base_routes(start, end, G):
    u = ox.nearest_nodes(G, start[1], start[0])
    v = ox.nearest_nodes(G, end[1], end[0])

    gen = nx.shortest_simple_paths(G, u, v, weight="travel_time_new")
    paths = list(islice(gen, 10))

    results = []

    for path in paths:
        base_time = 0.0
        for i in range(len(path) - 1):
            edge = G.get_edge_data(path[i], path[i+1])
            base_time += edge.get("travel_time_new", 1.0)

        results.append({
            "path": path,
            "coords": [[G.nodes[n]["y"], G.nodes[n]["x"]] for n in path],
            "base_time": base_time
        })

    return results

base_routes = get_10_base_routes(RITZ_CARLTON, KHAN_SHATYR, G_simple)

In [69]:
def get_5_penalized_routes(start, end, G):
    u = ox.nearest_nodes(G, start[1], start[0])
    v = ox.nearest_nodes(G, end[1], end[0])

    gen = nx.shortest_simple_paths(G, u, v, weight="travel_time_new")
    paths = list(islice(gen, 10))

    evaluated = []

    for path in paths:
        res = evaluate_path_with_penalties(path, G)
        evaluated.append(res)

    evaluated.sort(key=lambda x: x["total_time"])

    return evaluated[:5]

penalized_routes = get_5_penalized_routes(RITZ_CARLTON, KHAN_SHATYR, G_simple)

AttributeError: 'int' object has no attribute 'get'